# Lab - BUILD: ShopAssist Extracts Return Requests

One working function is a demo; a batch with a status per record is something the
support team can use. Three tasks put the checked extraction to work on a folder of
emails.

| Task | What you learn |
| --- | --- |
| 1 | Run the same checked extraction over every file in a folder |
| 2 | Keep a status per record - missing values go to a person |
| 3 | Save the batch as JSON and count what needs review |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

It writes the five emails into an `emails` folder - the support inbox export - and
defines `extract()`: the schema, the validation and the retry you built in the previous
lab. In the video the same function is imported from `validate.py`.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# --- The support inbox export: five emails, one text file each (provided) ---
INBOX = {
    "001.txt": "Hi, I want to return the blue jacket from order ORD-12345678.\n"
               "It arrived last week but it is too small. Thanks, Alex\n",
    "002.txt": "Hello, the shoes I got are damaged. Please tell me how to send\n"
               "them back. Sam\n",
    "003.txt": "Order ORD-87654321: the backpack is bigger than I expected and\n"
               "I would like to return it. Best, Priya\n",
    "004.txt": "The headphones from order ORD-7788 stopped working after two\n"
               "days. I want a return. Kim\n",
    "005.txt": "I ordered a lamp (ORD-11112222) as a gift but the recipient\n"
               "already has one. Can I send it back? Jordan\n",
}
Path("emails").mkdir(exist_ok=True)
for name, text in INBOX.items():
    (Path("emails") / name).write_text(text)

# --- extract(): the checked extraction you built in the previous lab (provided) ---
INSTRUCTION = (
    "Extract the return request from this customer email. "
    "Order IDs look like ORD-12345678. If the email gives no "
    "order number, set order_id to null. Never invent one."
)
SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": ["string", "null"]},
        "item": {"type": "string"},
        "reason": {"type": "string", "enum": ["too_small", "too_big", "damaged", "other"]},
    },
    "required": ["order_id", "item", "reason"],
    "additionalProperties": False,
}


def validate(data):
    order_id = data["order_id"]
    if order_id is not None and not re.fullmatch(r"ORD-\d{8}", order_id):
        return f"order_id {order_id!r} is not a valid order ID"
    return None


def extract(email):
    messages = [{"role": "user", "content": INSTRUCTION + "\n\n" + email}]
    for attempt in range(3):
        response = client.messages.create(
            model=MODEL, max_tokens=1000,
            output_config={"effort": "low",
                           "format": {"type": "json_schema", "schema": SCHEMA}},
            messages=messages)
        for block in response.content:
            if block.type == "text":
                text = block.text
        data = json.loads(text)
        error = validate(data)
        if error is None:
            return data
        print("  attempt", attempt + 1, "rejected:", error)
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": error + ". Fix it, or use null."})
    return None


print("emails/:", sorted(p.name for p in Path("emails").glob("*.txt")))

---

## Task 1 - Run the extraction over the folder

In [ ]:
# ============================================================
# TASK 1 - Run the extraction over the folder
# ============================================================
#
# WHAT TO DO
#   Walk through the email files in order, read each one,
#   extract its fields, and note which file it came from.
#
# WHY IT MATTERS
#   Every email goes through the same path you already built:
#   the schema, the validation and the retry. Nothing new is
#   trusted - the batch only repeats a checked step.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the script".
#
# HOW TO DO IT
#   Two blanks:
#
#       "*.txt"            the files to pick from the folder
#       path.read_text()   the email's text, read from the file
# ============================================================

requests = []

try:
    # BEGIN SOLUTION
    for path in sorted(Path("emails").glob("*.txt")):
        data = extract(path.read_text())
        data["email"] = path.name
        requests.append(data)
    # SCAFFOLD: for path in sorted(Path("emails").glob(...)):   # "*.txt" - every email file
    # SCAFFOLD:     data = extract(...)                          # path.read_text() - the email's text
    # SCAFFOLD:     data["email"] = path.name
    # SCAFFOLD:     requests.append(data)
    # END SOLUTION: replace each ... below with the value named beside it
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

for r in requests:
    print(r["email"], r)

check("read_folder", requests=requests)

---

## Task 2 - A status for every record

In [ ]:
# ============================================================
# TASK 2 - A status for every record
# ============================================================
#
# WHAT TO DO
#   Give every record a status: "ok" when it has an order ID,
#   "needs review" when the order ID is null. Print one line per
#   email.
#
# WHY IT MATTERS
#   It is tempting to send the unclear emails back to Claude
#   with more pressure until it produces an order number. It
#   would produce one, and it would be wrong. A missing value is
#   information: it tells the team which emails a person must
#   read.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then we add two fields of our own".
#
# HOW TO DO IT
#   Two blanks, the two statuses in quotes:
#
#       "ok"             there is an order ID
#       "needs review"   there is none - a person reads it
# ============================================================

for data in requests:
    # BEGIN SOLUTION
    data["status"] = "ok" if data["order_id"] else "needs review"
    # SCAFFOLD: data["status"] = ... if data["order_id"] else ...   # "ok", then "needs review"
    # END SOLUTION: replace each ... below with the status named beside it
    print(f"{data['email']}  {str(data['order_id']):14} "
          f"{data['item']:12} {data['reason']:10} {data['status']}")

check("status", requests=requests)

---

## Task 3 - Save the batch and count the reviews

In [ ]:
# ============================================================
# TASK 3 - Save the batch and count the reviews
# ============================================================
#
# WHAT TO DO
#   Save every record to return_requests.json, then count how
#   many need a person.
#
# WHY IT MATTERS
#   The JSON file holds all five, ready for the order system or
#   for a person. The count is what the support team looks at
#   first: how many emails today could not be handled without
#   someone reading them.
#
# WHERE TO SEE IT IN THE LECTURE
#   "At the end, we save the list as a JSON file".
#
# HOW TO DO IT
#   Two blanks:
#
#       "return_requests.json"   the file name, in quotes
#       "ok"                     a record needs review when its
#                                status is not this
# ============================================================

try:
    # BEGIN SOLUTION
    Path("return_requests.json").write_text(json.dumps(requests, indent=2))
    need_review = sum(r["status"] != "ok" for r in requests)
    # SCAFFOLD: Path(...).write_text(json.dumps(requests, indent=2))   # "return_requests.json"
    # SCAFFOLD: need_review = sum(r["status"] != ... for r in requests)            # "ok"
    # END SOLUTION: replace each ... below with the value named beside it
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

saved = json.loads(Path("return_requests.json").read_text())
print(len(saved), "requests saved,", need_review, "need review")

check("save", saved=saved, need_review=need_review)

---

## Done

Run the checked extraction over the whole batch, keep a status per record, and route
the records with missing values to a person instead of forcing a value.